In [1]:
from hydromt.readers import read_workflow_yaml
from hydromt_sfincs import SfincsModel

In [2]:
fn_config = r"C:\PhD\SFINCS\SFINCS_cloned\hydromt_sfincs\examples\sfincs_base_update.yml"

In [3]:
modeltype, kwargs, steps = read_workflow_yaml(fn_config, modeltype="sfincs")


In [10]:
steps[0]

{'observation_points.create': {'locations': WindowsPath('C:/PhD/SFINCS/SFINCS_cloned/hydromt_sfincs/examples/data/compound_example_observation_points.geojson')}}

In [ ]:
mod = SfincsModel(root=r"C:\PhD\SFINCS\output\sfinxcs_620947", mode="r"
                  )
mod.update(steps=steps)

In [11]:
from hydromt.readers import read_yaml

In [17]:
test = read_yaml(r"C:\PhD\SFINCS\SFINCS_cloned\hydromt_sfincs\delta_model\code\scenarios.yml")
test

{'global': {'data_libs': ['data_catalog_v1']},
 'scenarios': {'baseline': ['use this to plot the baselines and find where to make restart file'],
  'river_flood': {'steps': [{'config.update': {'rsrtfile': 'C:/PhD/SFINCS/output/sfincs_620947/sfincs.20181209.000000.rst'}},
    {'discharge_points.create_timeseries': {'index': [0],
      'shape': 'gaussian',
      'peak': 700,
      'timestep': 600}}]},
  'storm_surge': [{'config.update': {'rsrtfile': 'C:/PhD/SFINCS/output/sfincs_620947/sfincs.20181209.000000.rst'}},
   {'water_level.create_timeseries': {'shape': 'gaussian',
     'timestep': 600,
     'offset': 0.5,
     'peak': 3}}]}}

In [27]:
scenarios = test["scenarios"].keys()

In [28]:
scenarios

dict_keys(['baseline', 'river_flood', 'storm_surge'])

In [31]:
scenario_1 = test["scenarios"]["river_flood"]
steps_1 = scenario_1["steps"]
steps_1

[{'config.update': {'rsrtfile': 'C:/PhD/SFINCS/output/sfincs_620947/sfincs.20181209.000000.rst'}},
 {'discharge_points.create_timeseries': {'index': [0],
   'shape': 'gaussian',
   'peak': 700,
   'timestep': 600}}]

In [ ]:
# baseline_root = ...
# for scenario in scenarios:
#     new_root = ...
#     steps = test["scenarios"][scenario]["steps"]

#     # First read the model
#     mod = SfincsModel(root=baseline_root, mode="r")
#     mod.read()

#     # Now change the root of the model
#     mod.root.set(new_root, mode="w+")

#     # Now update the model
#     # mod.update(steps=steps_1)

#     mod.config.update(**steps["config.update"])
#     mod.discharge_points.create_timeseries(**steps["discharge_points.create_timeseries"])

In [ ]:
# Only keeps Q2 at entry domain into model 

import geopandas as gpd
from tqdm import tqdm

# 1. Load data
target_basin = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\4_delta_polygons.geojson")
target_basin = target_basin[target_basin['BasinID2'] == 4267691].to_crs(epsg=3857)

sword = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\Global_rivers.gpkg", mask=target_basin).to_crs(epsg=3857)
lin_points = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\rivers_lin.gpkg", mask=target_basin).to_crs(epsg=3857)

# 2. Find where SWORD intersects the boundary (The Inflow/Outflow points)
boundary_line = target_basin.geometry.boundary.iloc[0]
sword['intersects_boundary'] = sword.intersects(boundary_line)
inflow_reaches = sword[sword['intersects_boundary']].copy()

# 3. Join Lin Q2 to these specific boundary reaches
# We use a constrained join: closest point within 1km
print("Snapping Lin Q2 values to boundary inflow points...")
joined_inflows = gpd.sjoin_nearest(inflow_reaches, lin_points, max_distance=1000, how='inner')

# 4. Create a lookup for the boundary reaches
q2_boundary_map = dict(zip(joined_inflows['reach_id'], joined_inflows['Q2']))

# 5. Apply back to your SWORD dataset
sword['inflow_Q2'] = sword['reach_id'].map(q2_boundary_map)

print(f"Assigned Q2 to {len(q2_boundary_map)} boundary reaches.")

print(sword[['reach_id', 'inflow_Q2', 'width']].dropna())

# Calculate depth (h) based on bankfull discharge (Q) - from Dirk's paper but # From Leopold and Maddock (1953) says that this equation calculates width 
a = 0.27  
b = 0.30
sword['rivdph'] = a * (sword['inflow_Q2']**b) 

print(sword[['reach_id', 'inflow_Q2','width', 'rivdph']].dropna())

Snapping Lin Q2 values to boundary inflow points...
Assigned Q2 to 4 boundary reaches.


In [3]:
import geopandas as gpd

# 1. Load data (Keep your existing loading logic)
target_basin = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\4_delta_polygons.geojson")
target_basin = target_basin[target_basin['BasinID2'] == 620947].to_crs(epsg=3857)

sword = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\Global_rivers.gpkg", mask=target_basin).to_crs(epsg=3857)
lin_points = gpd.read_file(r"C:\PhD\SFINCS\SFINCS_cloned\input\rivers_lin.gpkg", mask=target_basin).to_crs(epsg=3857)

# We join 'lin_points' attributes to the 'sword' geometry
print("Snapping Lin Q2 values to all SWORD segments...")
sword_with_q2 = gpd.sjoin_nearest(
    sword,                          # width, slope
    lin_points,                     # Q2, width_m  
    max_distance=1000, 
    how='left'
)

print(f"Assigned Q2 values to {sword_with_q2['Q2'].notnull().sum()} segments out of {len(sword_with_q2)} total.")

Snapping Lin Q2 values to all SWORD segments...
Assigned Q2 values to 2 segments out of 3 total.


In [9]:
print(sword_with_q2[['Q2', 'width_m', 'slope', 'width']])

           Q2    width_m     slope  width
0  322.661091  152.28017  0.031398  216.0
1  322.661091  152.28017  0.063883  366.0
2         NaN        NaN  0.000000  351.0


In [10]:
# Calculate depth (h) based on bankfull discharge (Q) - from Dirk's paper but # From Leopold and Maddock (1953) says that this equation calculates width 
a = 0.27  
b = 0.30
sword_with_q2['rivdph'] = a * (sword_with_q2['Q2']**b) 

print(sword_with_q2[['Q2', 'width_m', 'slope', 'width', 'rivdph']])


           Q2    width_m     slope  width    rivdph
0  322.661091  152.28017  0.031398  216.0  1.527523
1  322.661091  152.28017  0.063883  366.0  1.527523
2         NaN        NaN  0.000000  351.0       NaN


In [84]:
# Save to Shapefile
output_shp = r"C:\PhD\SFINCS\SFINCS_cloned\input\SWORD_Q2_export_aus.shp"
sword_with_q2.to_file(output_shp)

print(f"Exported to Shapefile: {output_shp}")

Exported to Shapefile: C:\PhD\SFINCS\SFINCS_cloned\input\SWORD_Q2_export_aus.shp


In [26]:
# Width calculation = a * Q^b --> power-law equation --> Leopold and Maddock (1953) and Andreadis et al (2013) 
a = 7.2
b = 0.50
sword_with_q2["rivwth"] = (a * (sword_with_q2["Q2"].astype(float) ** b)).astype(float)
# Keep the better width data from SWORD (variable = 'width) if available, otherwise keep calculated value from power-law relationship 
sword_with_q2.loc[sword_with_q2["width"].notna(), "rivwth"] = (sword_with_q2["width"])
# # Check if there are any NaN values in the rivwth column
# missing_rivwth = sword_with_q2["rivwth"].isna().sum()
# print(f"Number of missing rivwth values after filling: {missing_rivwth}")
import numpy as np
min_riv_slope = 1e-6
# If the slope is NaN OR if it's too small, set it to min_riv_slope
sword_with_q2["slope"] = np.where(
    (sword_with_q2["slope"].isna()) | (sword_with_q2["slope"] < min_riv_slope),
    min_riv_slope,
    sword_with_q2["slope"]
)
# Depth calculation - choose which method to prioritise to calculate river depth (rivdph) --> Mannings or power law 
depth_calculation = "manning"

if depth_calculation == "manning":
    sword_with_q2["rivdph"] = (
        (0.030 * sword_with_q2["Q2"])
        / (np.sqrt(sword_with_q2["slope"]) * sword_with_q2["rivwth"])
    ) ** (3 / 5)
elif depth_calculation == "power_law": # Andreadis et al. (2013)
    c = 0.27
    f = 0.30  
    sword_with_q2["rivdph"] = c * (sword_with_q2["Q2"].astype(float) ** f)
# Replace 'rivdph' values with the minimum value where they are less than the minimum
min_rivdph = 0 # Note: Making this value higher or lower can affect results
sword_with_q2["rivdph"] = (np.where(
    sword_with_q2["rivdph"] < min_rivdph, min_rivdph, sword_with_q2["rivdph"]
)).astype(float)


print(sword_with_q2[['Q2', 'width_m', 'slope', 'width', 'rivdph', 'rivwth']])

           Q2    width_m     slope  width    rivdph  rivwth
0  322.661091  152.28017  0.031398  216.0  0.438305   216.0
1  322.661091  152.28017  0.063883  366.0  0.258117   366.0
2         NaN        NaN  0.000010  351.0       NaN   351.0


In [ ]:
# OBSERVATION POINTS 

import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

# 1. Load data
rivers = sword_with_q2  # Use the dataset with Q2 values assigned

# 2. Merge all segments into one single "LineString" or "MultiLineString"
# This removes overlaps and treats the whole set as one system
combined_rivers = unary_union(rivers.geometry)
total_length = combined_rivers.length

# 3. Calculate the interval needed for exactly 20 points
# We use 19 intervals to get 20 points (point at 0 and point at the end)
interval = total_length / 19

points_list = []
current_dist = 0

for i in range(20):
    # Interpolate a point at the calculated distance
    pt = combined_rivers.interpolate(current_dist)
    points_list.append({'obs_id': i + 1, 'geometry': pt})
    current_dist += interval

# 4. Save to a clean file
obs_points = gpd.GeoDataFrame(points_list, crs=rivers.crs)
